<a href="https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule is to prioritize webpages that may need a content refresh. A page gets a higher score when it is old, has not been updated recently, has meaningful search impressions, or has a weaker average search position.

Reason codes:
- OLD_CONTENT: the content is older than 180 days.
- NOT_UPDATED: the page has not been updated for at least 180 days.
- HIGH_IMPRESSIONS: the page has at least 500 impressions in the last 90 days.
- WEAK_POSITION: the page has a relatively weak average search position.

The score is only a simple baseline for prioritization. It is not a prediction or proof that a page needs updating.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np

if not os.path.exists("flyrank-ml-internship-starter"):
    subprocess.run([
        "git", "clone",
        "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    ], check=True)

df = pd.read_csv(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [2]:
df["baseline_score"] = (
    (df["content_age_days"] >= 180).astype(int) * 2
    + (df["days_since_last_update"] >= 180).astype(int) * 2
    + (df["impressions_90d"] >= 500).astype(int) * 2
    + (df["avg_position"] > 10).astype(int) * 1
)

df["reason_codes"] = ""

df.loc[
    df["content_age_days"] >= 180,
    "reason_codes"
] += "OLD_CONTENT;"

df.loc[
    df["days_since_last_update"] >= 180,
    "reason_codes"
] += "NOT_UPDATED;"

df.loc[
    df["impressions_90d"] >= 500,
    "reason_codes"
] += "HIGH_IMPRESSIONS;"

df.loc[
    df["avg_position"] > 10,
    "reason_codes"
] += "WEAK_POSITION;"

queue = df.sort_values(
    "baseline_score",
    ascending=False
).copy()

queue["rank"] = range(1, len(queue) + 1)

output_columns = [
    "rank",
    "baseline_score",
    "reason_codes",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

queue[output_columns].to_csv(
    "baseline_action_score.csv",
    index=False
)

print("Queue created.")
print("Rows:", len(queue))
print(queue[output_columns].head(10))

Queue created.
Rows: 30000
       rank  baseline_score  \
16751     1               7   
21268     2               7   
23215     3               7   
26799     4               7   
698       5               7   
12045     6               7   
16514     7               7   
20837     8               7   
7021      9               7   
11630    10               7   

                                            reason_codes  content_age_days  \
16751  OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...               231   
21268  OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...               231   
23215  OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...               231   
26799  OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...               231   
698    OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...               231   
12045  OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...               231   
16514  OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...               231   
20837  

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 pages from the baseline queue. For each page, the action is to review the page for a possible content refresh. The reason code explains why the page received a high score. The confidence is limited because this is a simple rule, so each recommendation should be manually checked before taking action. A recommendation could be wrong if the page is intentionally old, already performs well, or has a reason unrelated to content quality.

In [3]:
top20 = queue.head(20).copy()

top20[
    [
        "rank",
        "baseline_score",
        "reason_codes",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
]

,rank,baseline_score,reason_codes,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr
16751,1,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,194,61678,19.7,0.15
21268,2,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,193,13299,10.5,0.49
23215,3,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,194,1316,21.8,0.15
26799,4,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,194,828,18.6,0.24
698,5,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,194,4590,31.0,0.00
12045,6,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,193,7558,17.9,0.20
16514,7,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,194,59472,24.8,0.13
20837,8,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,193,1697,15.8,0.12
7021,9,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,231,194,25715,22.2,0.23
11630,10,7,OLD_CONTENT;NOT_UPDATED;HIGH_IMPRESSIONS;WEAK_...,232,183,545,17.8,0.18


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some high-ranked pages may be weak picks because age or impressions alone do not mean that the content needs to be refreshed. A page may be intentionally old but still useful and performing well. The baseline should therefore be treated as a review queue rather than an automatic update decision.

For leakage, I did not use trend_direction or trend_pct in the baseline score. I also did not use product decision flags or future outcome information. The score uses only available page and historical search signals.

In [4]:
score_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position"
]

leakage_fields = [
    "trend_direction",
    "trend_pct"
]

print("Features used for baseline:")
print(score_features)

print("\nLeakage fields:")
for col in leakage_fields:
    print(col, "used:", col in score_features)

Features used for baseline:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position']

Leakage fields:
trend_direction used: False
trend_pct used: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.